# 2.8c — Borne + Témoin extrémal + Concentration : ce que `learning_theory_lean` sait déjà faire

**Navigation** : [<< 2.8b-Theorie-PAC-Lean](2.8b-Theorie-PAC-Lean.ipynb) | [Index](../README.md)

**Kernel** : Python 3 (cpu)

**Compagnon formel** : `MyIA.AI.Notebooks/ML/learning_theory_lean/`

---

## Concept

Trois temps, qui distinguent une borne décorative d'une borne **utile** :

```
BORNE          -- une quantite ne peut pas depasser X
TEMOIN EXTR.   -- voici un objet qui atteint X (la borne est serree, pas decorative)
CONCENTRATION  -- voici avec quelle probabilite on s'en ecarte
```

Le triptyque reapparait dans tout le machine learning :
la **borne PAC** est inutile tant qu'on n'a pas son temoin extremal ;
et la **concentration** dit *combien d'echantillons* pour etre proche.

Sur le perceptron de Novikoff, le lake `learning_theory_lean` montre :

- **BORNE** (`Perceptron.Convergence.lean`) : `n <= (R/gamma)^2` mises a jour.
  Deux lemmes : alignement (Lem A) et norme (Lem B), combines par Cauchy-Schwarz.
- **TEMOIN** (`Perceptron.Tightness.lean`) : `witnessPts = [1+I, 1-I]`, separateur `u = 1`,
  `gamma = 1`, `R = sqrt(2)`. Apres 2 mises a jour, `n * gamma^2 = R^2` **exactement**.
- **CONCENTRATION** (`PacLearning.Hoeffding.lean`) : `P[|emp - true| > eps] <= 2 * exp(-2 n eps^2)`.

Ce notebook **consomme** ces theoremes (file:ligne), il ne les re-prouve pas.
Il mesure leur application sur des instances explicites.

In [1]:
import numpy as np
from pathlib import Path
from typing import List, Tuple

RNG = np.random.default_rng(seed=20260822)
print(f'numpy={np.__version__}')


numpy=2.4.2


## Section 1 -- Reconstruction de la borne (Novikoff jouet)

**But** : reproduire le perceptron sur des donnees lineairement separables, compter
les erreurs `n`, et confronter `n` a la borne `(R/gamma)^2`.

**Certificat** : `PerceptronRun.norm_bound` (Convergence.lean:80) + `align_growth` (Convergence.lean:43)
donnent l'enveloppe `n * gamma^2 <= R^2`.

In [2]:
def perceptron_run(pts: np.ndarray, lbl: np.ndarray, u: np.ndarray) -> Tuple[int, list]:
    """Perceptron classique sur points 2D. Retourne (n_mistakes, trace)."""
    n = len(pts)
    w = np.zeros(2)
    trace = []
    for k in range(n):
        x = pts[k]
        y = lbl[k]
        # Erreur : prediction y*(w . x) <= 0
        if y * np.dot(w, x) <= 0:
            w = w + y * x
            trace.append((k, w.copy()))
    return len(trace), trace

def gamma_R(pts: np.ndarray, lbl: np.ndarray, u: np.ndarray) -> Tuple[float, float]:
    """Marge et rayon sur les donnees."""
    margins = lbl * (pts @ u)
    gamma = float(margins.min())           # plus petite marge sur les donnees
    R = float(np.linalg.norm(pts, axis=1).max())  # plus grand rayon
    return gamma, R


In [3]:
# Domaine jouet : 2D, separateur canonique u = (1, 0) (i.e. label = signe(x[0]))
# Donnees dans la bande -2 <= x[0] <= 2, -1 <= x[1] <= 1
n_dataset = 80
pts = RNG.uniform(low=[-2, -1], high=[2, 1], size=(n_dataset, 2))
u = np.array([1.0, 0.0])
lbl = np.sign(pts @ u).astype(int)
lbl[lbl == 0] = 1   # eviter 0 sur les points sur l'axe

gamma, R = gamma_R(pts, lbl, u)
n_mistakes, trace = perceptron_run(pts, lbl, u)
bound = (R / gamma) ** 2
print(f'n={n_mistakes} erreurs, (R/gamma)^2={bound:.2f}, n <= (R/gamma)^2 ? {n_mistakes <= bound}')
print(f'gamma={gamma:.4f}, R={R:.4f}, ratio R/gamma={R/gamma:.4f}')


n=4 erreurs, (R/gamma)^2=16184.12, n <= (R/gamma)^2 ? True
gamma=0.0163, R=2.0723, ratio R/gamma=127.2168


### Lecture du resultat

**Mesure** : `n <= (R/gamma)^2`. La borne est respectee (c'est la proposition que Lean prouve,
et on la voit numeriquement). Ce n'est pas une demonstration : c'est une **verification** que
l'implementation Python respecte la proposition formelle.

**Pourquoi une borne si large ?** Sur ce domaine jouet, `gamma` est proche de 0 (le point le plus
proche du separateur est presque dessus) et `R` est de l'ordre de 2. Donc `(R/gamma)^2 >> 1`,
et `n << borne`. La borne est utile **au sens ou elle borne**, pas au sens ou elle est precise.
La section 2 exhibe un cas ou elle est **exactement** atteinte.

## Section 2 -- Temoin extremal (egalite `n*gamma^2 = R^2`)

**Certificat** : `Perceptron.Tightness.lean:139` (`tightnessRun_saturates`) et
`Tightness.lean:153` (`novikoff_bound_is_sharp`).
Le temoin explicite est : `witnessPts = [1+I, 1-I]` (Tightness.lean:49), labels = `+1`
sur les deux points, separateur `u = 1`, marge `gamma = 1`, rayon `R = sqrt(2)`.

**But** : reproduire le temoin en Python, verifier que `n * gamma^2 = R^2` exactement
(egalite, pas quasi-egalite racontee).

In [4]:
# Temoin explicite du lake (Tightness.lean:49) :
# witnessPts 0 = 1+I, witnessPts 1 = 1-I
# witnessLbl _ = 1
# Separateur u = 1 (canonique dans C vu comme R^2)

# Convention : on travaille dans R^2 (re, im) plutot que dans C
witness_pts = np.array([[1.0, 1.0], [1.0, -1.0]])
witness_lbl = np.array([1, 1])
u_w = np.array([1.0, 0.0])  # separateur "x[0] > 0"

gamma_w, R_w = gamma_R(witness_pts, witness_lbl, u_w)
n_w, trace_w = perceptron_run(witness_pts, witness_lbl, u_w)

lhs = n_w * gamma_w ** 2
rhs = R_w ** 2
print(f'n={n_w}, gamma={gamma_w}, R={R_w}')
print(f'n * gamma^2 = {lhs}')
print(f'R^2 = {rhs}')
print(f'Egalite n*gamma^2 = R^2 ? {np.isclose(lhs, rhs)}')


n=2, gamma=1.0, R=1.4142135623730951
n * gamma^2 = 2.0
R^2 = 2.0000000000000004
Egalite n*gamma^2 = R^2 ? True


### Lecture du temoin

**Verifie** : `n * gamma^2 == R^2` a epsilon machine pres.
C'est ce que Lean certifie dans `tightnessRun_saturates` :
**la borne n'est pas decorative, elle est atteinte**.

**Consequence** : aucune constante strictement plus petite que `1` devant `(R/gamma)^2`
ne peut etre universelle (Tightness.lean:153 `novikoff_bound_is_sharp`).

Une borne sans temoin extremal est un majorant ;
une borne **avec son temoin extremal** est une **caracterisation** :
la difference entre *on n'a pas trouve mieux* et *il n'y a pas mieux*.

## Section 3 -- Concentration (Hoeffding bilateral)

**Certificat** : `PacLearning.Hoeffding.lean:318` (`hoeffding_concentration`).

**Enonce** : pour `n >= 1` tirages i.i.d. d'indicateurs `X_i in {0, 1}`,
`P[|empError - trueError| > eps] <= 2 * exp(-2 n eps^2)`.

**But** : sur un echantillon simule, mesurer l'ecart |emp - true| et confronter
la queue de distribution a la borne de Hoeffding.

In [5]:
# Tirage : on simule une probabilite vraie p_true = 0.3, on tire n echantillons
# de Bernoulli et on regarde l'ecart |emp - p_true| sur N repetitions.
p_true = 0.3
n_per_rep = 200
n_reps = 5000

samples = RNG.binomial(n=n_per_rep, p=p_true, size=n_reps) / n_per_rep
abs_dev = np.abs(samples - p_true)

# Borne de Hoeffding pour chaque eps
eps_grid = np.array([0.05, 0.10, 0.15])
hoeff_bound = 2 * np.exp(-2 * n_per_rep * eps_grid ** 2)

print(f'p_true={p_true}, n={n_per_rep}, N_reps={n_reps}')
print(f'Ecart empirique moyen = {abs_dev.mean():.4f}')
print(f'Ecart max observe = {abs_dev.max():.4f}')
print()
print('eps | P[emp-true|>eps] empirique | Hoeffding')
print('-' * 55)
for eps in eps_grid:
    p_emp = (abs_dev > eps).mean()
    p_h = 2 * np.exp(-2 * n_per_rep * eps ** 2)
    print(f'{eps:.2f} | {p_emp:.4f}                  | <= {p_h:.4f}')


p_true=0.3, n=200, N_reps=5000
Ecart empirique moyen = 0.0252
Ecart max observe = 0.1150

eps | P[emp-true|>eps] empirique | Hoeffding
-------------------------------------------------------
0.05 | 0.0946                  | <= 0.7358
0.10 | 0.0012                  | <= 0.0366
0.15 | 0.0000                  | <= 0.0002


### Lecture de la concentration

**Mesure** : pour chaque `eps`, la frequence empirique des ecarts superieurs est **bien en-dessous**
de la borne Hoeffding. C'est attendu : la borne est un majorant universel, pas une estimation.
La **vraie question pedagogique** est :

1. La borne est-elle **respectee** a chaque niveau d'epsilon ? (Oui, par construction theorique.)
2. La borne est-elle **serree** a un epsilon particulier ? (Rarement aux grands n --
   la borne de Chernoff exacte via MGF est plus precise. Hoeffding est une borne **simple**
   qui donne une intuition claire de la dependance en `n` et `eps`.)

**Conclusion pedagogique** : Hoeffding dit qu'on converge en `O(1/sqrt(n))` en probabilite.
Pour un ecart desire `eps`, il faut `n ~ O(1/eps^2)` tirages.
La **borne PAC** `pac_finite_class_bound` (PacFiniteBound.lean:377) specialise cela aux classes
d'hypotheses finies en ajoutant un facteur `log|H|` par union bound.

## Conclusion -- Le triptyque dans le machine learning

Les **trois temps** que ce notebook a traverses sont recurrents :

| Triptyque | Borne | Temoin extremal | Concentration |
|-----------|-------|-----------------|---------------|
| **Novikoff perceptron** | `n <= (R/gamma)^2` (Convergence.lean) | `n*gamma^2 = R^2` (Tightness.lean) | n/a (algorithme online) |
| **Hoeffding bilateral**  | `P[|emp-true|>eps] <= 2 exp(-2n eps^2)` (Hoeffding.lean) | `bernoulli_subgaussian` (MGF.lean) | convergence en `O(1/sqrt(n))` |
| **PAC fini**             | `n >= (log|H| + log(1/delta)) / eps` (PacFiniteBound.lean) | concept de VC-dim (non formalise ici) | union bound sur `|H|` |

**Le lake comme interprete certifie** : on **consomme** la preuve formelle (le fichier Lean donne
le `file:line` du theoreme), on **mesure** sur des instances explicites, et on **discute** quand
la borne est decorative vs informante.

**Limites du notebook** :

- `lake build SUCCESS` du module `learning_theory_lean` non re-verifie dans ce cycle
  (cache `~/.lake` peut etre orphelin). On s'appuie sur la sortie du compteur de `sorry`
  dans le body PR.
- Le temoin de Hoeffding (`bernoulli_subgaussian`) demande un import `Mathlib.Probability`
  qu'on ne fait pas ici : on **cite** le fichier, on ne le rejoue pas.
- La VC-dimension (generalisation au cas infini) est en dehors du perimetre de ce notebook.

**Suite suggeree** : un notebook 2.8d sur la VC-dimension, qui sort du cas fini pour attaquer
le cas `|H| = infini`. Cela necessiterait probablement un nouveau module Lean ou l'import de
`Mathlib.Probability.Martingale.Basic`.

In [6]:
# Verification rapide : les theoremes cites existent-ils dans le lake ?
from pathlib import Path
lean_root = Path('MyIA.AI.Notebooks/ML/learning_theory_lean')

theorems_to_check = [
    ('Perceptron/Convergence.lean', 'PerceptronRun.align_growth'),
    ('Perceptron/Convergence.lean', 'PerceptronRun.norm_bound'),
    ('Perceptron/Convergence.lean', 'novikoff_mistake_bound'),
    ('Perceptron/Tightness.lean', 'witnessPts'),
    ('Perceptron/Tightness.lean', 'tightnessRun_saturates'),
    ('Perceptron/Tightness.lean', 'novikoff_bound_is_sharp'),
    ('PacLearning/Hoeffding.lean', 'hoeffding_concentration'),
    ('PacLearning/Hoeffding.lean', 'hoeffding_upper_tail'),
    ('PacLearning/PacFiniteBound.lean', 'pac_finite_class_bound'),
]

print(f'{"file":<40} {"theorem":<35} found?')
print('-' * 85)
for fpath, tname in theorems_to_check:
    full = lean_root / fpath
    if not full.exists():
        print(f'{fpath:<40} {tname:<35} FILE MISSING')
        continue
    text = full.read_text()
    found = tname in text
    print(f'{fpath:<40} {tname:<35} {"OK" if found else "MISSING"}')


file                                     theorem                             found?
-------------------------------------------------------------------------------------
Perceptron/Convergence.lean              PerceptronRun.align_growth          OK
Perceptron/Convergence.lean              PerceptronRun.norm_bound            OK
Perceptron/Convergence.lean              novikoff_mistake_bound              OK
Perceptron/Tightness.lean                witnessPts                          OK
Perceptron/Tightness.lean                tightnessRun_saturates              OK
Perceptron/Tightness.lean                novikoff_bound_is_sharp             OK
PacLearning/Hoeffding.lean               hoeffding_concentration             OK
PacLearning/Hoeffding.lean               hoeffding_upper_tail                OK
PacLearning/PacFiniteBound.lean          pac_finite_class_bound              OK


In [7]:
# Verifier que les modules cites ont 0 sorry (preuve reelle, pas stub)
import subprocess, json
result = subprocess.run(
    ['python', 'scripts/lean/count_code_sorry.py', '--json'],
    capture_output=True, text=True, cwd='.',
)
if result.returncode != 0:
    print('count_code_sorry not available, fallback grep')
    import re
    for fpath in [t[0] for t in theorems_to_check]:
        full = lean_root / fpath
        text = full.read_text()
        # Filtrer les '-- commentaires' (ne matchent pas 'exact sorry')
        sorry_matches = [l for l in text.splitlines() if re.search(r'\bsorry\b', l)]
        code_sorry = [l for l in sorry_matches if not l.strip().startswith('--') and '/--' not in l]
        print(f'{fpath}: {len(code_sorry)} sorry code (approx)')
else:
    data = json.loads(result.stdout)
    print(json.dumps(data, indent=2)[:500])


{
  "lakes": [
    {
      "lake": "MyIA.AI.Notebooks/GameTheory/conway_cgt_lean",
      "files": 3,
      "naive_sorry": 0,
      "code_sorry": 0,
      "distinct_code_sorry": 0,
      "vacuous": []
    },
    {
      "lake": "MyIA.AI.Notebooks/GameTheory/game_theory_lean",
      "files": 47,
      "naive_sorry": 32,
      "code_sorry": 2,
      "distinct_code_sorry": 1,
      "vacuous": []
    },
    {
      "lake": "MyIA.AI.Notebooks/GameTheory/minimax_lean",
      "files": 9,
      "naive_so
